
# PE6201 A2 — D5 Live Model Runner

Use this notebook for **one member = one live battery**.

Expected package layout:

```text
<PACKAGE_ROOT>/
├── A2_scaffold/
├── A2_reference_data/
├── PE6201_D5_Live_Model_Runner.ipynb
└── D5_LIVE_MODEL_RUN_GUIDE.md
```

For a normal D5 final-model run:
- use **Problem B**
- use **live** backend
- use the frozen **40-case / 56-trial** battery
- use **v2**
- use **parallel** tool-call mode
- change only the assigned model and its matching prices

Only the teammate assigned to the controlled D2(b) comparison should use **v1**, and that v1 run should use the same model used by another teammate's v2 run.

This notebook measures the actual wall-clock runtime of the 56-trial battery and saves a model-specific copy of the result.


## Step 1 — Set the local package path

You only need to edit `PACKAGE_ROOT` below. The two folders `A2_scaffold` and `A2_reference_data` must be directly inside it.


In [1]:
from pathlib import Path
import os

# EDIT ONLY THIS PATH if your package is stored somewhere else.
# Running locally (no Google Drive): point this at the D5 folder that
# directly contains A2_scaffold/ and A2_reference_data/.
PACKAGE_ROOT = Path(r"D:\Github\PE6201_A2\D5\D5_live_model_runner")

SCAFFOLD = PACKAGE_ROOT / "A2_scaffold"
REFERENCE = PACKAGE_ROOT / "A2_reference_data"

print("PACKAGE_ROOT :", PACKAGE_ROOT)
print("A2_scaffold :", SCAFFOLD)
print("Reference   :", REFERENCE)


PACKAGE_ROOT : D:\Github\PE6201_A2\D5\D5_live_model_runner
A2_scaffold : D:\Github\PE6201_A2\D5\D5_live_model_runner\A2_scaffold
Reference   : D:\Github\PE6201_A2\D5\D5_live_model_runner\A2_reference_data



## Step 2 — Check that the required folders and files are present

This cell makes **no API calls**. It stops immediately if a required project file or Problem B data file is missing.


In [2]:

from pathlib import Path

required_scaffold = [
    "agent.py",
    "backends.py",
    "config.py",
    "guardrails.py",
    "harness.py",
    "prompt.py",
    "run_eval.py",
    "tools.py",
    "evaluation_cases.json",
]

required_reference = [
    "expected_outcomes_B.json",
    "data_B/referrals.json",
    "data_B/patients.json",
    "data_B/contacts.json",
    "data_B/specialties.json",
    "data_B/urgency_bands.json",
    "data_B/clinic_slots.json",
    "data_B/as_of.json",
]

missing = []

if not SCAFFOLD.is_dir():
    missing.append(str(SCAFFOLD))
if not REFERENCE.is_dir():
    missing.append(str(REFERENCE))

for rel in required_scaffold:
    if not (SCAFFOLD / rel).exists():
        missing.append(str(SCAFFOLD / rel))

for rel in required_reference:
    if not (REFERENCE / rel).exists():
        missing.append(str(REFERENCE / rel))

if missing:
    print("MISSING FILES/FOLDERS:")
    for p in missing:
        print(" -", p)
    raise FileNotFoundError(
        "Package check failed. Fix the folder structure before continuing."
    )

print("PASS: package structure looks correct.")
print("Scaffold files checked :", len(required_scaffold))
print("Reference files checked:", len(required_reference))


PASS: package structure looks correct.
Scaffold files checked : 9
Reference files checked: 8


## Step 3 — Load your own OpenRouter API key from `.env`

This repo ships a template at `.env.example` (repo root). To use it:
1. Copy `.env.example` to `.env` in the repo root (`PE6201_A2/.env`).
2. Replace `your_api_key_here` with your own OpenRouter key.
3. `.env` is gitignored — never commit it or paste the full key into `config.py`, this notebook, GitHub, or the result file.


In [ ]:
import os

# Repo root .env, two levels up from this D5 package folder.
ENV_PATH = PACKAGE_ROOT.parent.parent / ".env"

def load_dotenv(path):
    if not path.exists():
        return {}
    values = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        name, _, value = line.partition("=")
        values[name.strip()] = value.strip().strip('"').strip("'")
    return values

env_values = load_dotenv(ENV_PATH)
key = env_values.get("OPENROUTER_API_KEY") or os.environ.get("OPENROUTER_API_KEY")

if not key or key == "your_api_key_here":
    raise RuntimeError(
        f"OPENROUTER_API_KEY was not found. Copy {ENV_PATH.parent / '.env.example'} "
        f"to {ENV_PATH} and set your real key."
    )

key = key.strip()
os.environ["OPENROUTER_API_KEY"] = key

print("PASS: OpenRouter API key loaded from", ENV_PATH)
print("Starts with:", key[:6])
print("Length     :", len(key))
print("The full key is intentionally NOT printed.")



## Step 4 — Enter your assigned run settings

### Version
- `v2` = final interface. Use this for the normal D5 model battery.
- `v1` = old interface. Use this **only** if you are the teammate assigned to the one controlled D2(b) v1 comparison.

### Tool-call mode
- `parallel` = independent tool calls can share one model turn. This is the team's final D5 setting.
- `serial` = at most one tool call per model turn. Use only for the deliberate D2(c) comparison, not the normal D5 final battery.

### Prices
Enter the model's **input** and **output** prices in US dollars per **1,000,000 tokens**. Verify them against the current OpenRouter/vendor pricing page before running.


In [ ]:

# ===== EDIT THIS CELL FOR YOUR ASSIGNED RUN =====

MEMBER_NAME = "CHANGE_ME"

# OpenRouter model identifier, for example:
MODEL = "google/gemini-2.5-flash-lite"

# USD per 1,000,000 tokens.
PRICE_IN = 0.10
PRICE_OUT = 0.40

# Normal D5 members use "v2".
# Only the assigned D2(b) comparison member uses "v1".
RUN_VERSION = "v2"

# Final D5 battery should normally use "parallel".
TOOL_CALL_MODE = "parallel"

# ================================================

RUN_VERSION = RUN_VERSION.strip().lower()
TOOL_CALL_MODE = TOOL_CALL_MODE.strip().lower()

if MEMBER_NAME == "CHANGE_ME" or not MEMBER_NAME.strip():
    raise ValueError("Set MEMBER_NAME before continuing.")

if not MODEL.strip():
    raise ValueError("MODEL cannot be blank.")

if PRICE_IN < 0 or PRICE_OUT < 0:
    raise ValueError("Prices cannot be negative.")

if RUN_VERSION not in {"v1", "v2"}:
    raise ValueError("RUN_VERSION must be 'v1' or 'v2'.")

if TOOL_CALL_MODE not in {"serial", "parallel"}:
    raise ValueError("TOOL_CALL_MODE must be 'serial' or 'parallel'.")

print("Requested settings:")
print("  Member         :", MEMBER_NAME)
print("  Model          :", MODEL)
print("  Price input    : US$", PRICE_IN, "/ 1M tokens")
print("  Price output   : US$", PRICE_OUT, "/ 1M tokens")
print("  Version        :", RUN_VERSION)
print("  Tool-call mode :", TOOL_CALL_MODE)



## Step 5 — Apply the live model and prices to `config.py`

This cell changes only the run-specific configuration:
- `BACKEND = "live"`
- `MODEL`
- `PRICE_IN`
- `PRICE_OUT`

It also sets:
- `A2_DATA` to the supplied `A2_reference_data` folder
- `VERSION`
- `TOOL_CALL_MODE`

The notebook creates `config.py.before_d5_runner` the first time it edits the file.

Do **not** change the routing rules, prompt, agent, harness, evaluation set, or reference data between teammates.


In [ ]:

import os
import re
import shutil
from pathlib import Path

config_path = SCAFFOLD / "config.py"
backup_path = SCAFFOLD / "config.py.before_d5_runner"

if not backup_path.exists():
    shutil.copy2(config_path, backup_path)
    print("Created backup:", backup_path.name)
else:
    print("Backup already exists:", backup_path.name)

text = config_path.read_text(encoding="utf-8")

def replace_one(pattern, replacement, text, label):
    new_text, n = re.subn(pattern, replacement, text, count=1, flags=re.M)
    if n != 1:
        raise RuntimeError(f"Could not safely update {label} in config.py")
    return new_text

text = replace_one(
    r'^BACKEND\s*=\s*["\'][^"\']*["\'].*$',
    'BACKEND = "live"          # "scripted" | "live"',
    text,
    "BACKEND",
)

text = replace_one(
    r'^MODEL\s*=\s*["\'][^"\']*["\'].*$',
    f'MODEL = "{MODEL}"  # only used when BACKEND == "live"',
    text,
    "MODEL",
)

text = replace_one(
    r'^PRICE_IN\s*=\s*[0-9.eE+-]+\s*$',
    f'PRICE_IN = {PRICE_IN}',
    text,
    "PRICE_IN",
)

text = replace_one(
    r'^PRICE_OUT\s*=\s*[0-9.eE+-]+\s*$',
    f'PRICE_OUT = {PRICE_OUT}',
    text,
    "PRICE_OUT",
)

config_path.write_text(text, encoding="utf-8")

# These settings are intentionally environment-controlled by config.py.
os.environ["A2_DATA"] = str(REFERENCE)
os.environ["VERSION"] = RUN_VERSION
os.environ["TOOL_CALL_MODE"] = TOOL_CALL_MODE

print("PASS: run configuration applied.")



## Step 6 — Clear local Python caches before the run

This clears local `__pycache__`, `.pyc`, and already-imported project modules.

The actual battery is launched in a **new Python subprocess**, so every battery starts with fresh Python process state.

Note: this cannot disable an OpenRouter/provider-side prompt cache. If a provider reports cached tokens, those are recorded in the result and are part of the observed live-provider usage.


In [ ]:

import shutil
import sys
from pathlib import Path

removed_dirs = 0
removed_pyc = 0

for p in SCAFFOLD.rglob("__pycache__"):
    if p.is_dir():
        shutil.rmtree(p, ignore_errors=True)
        removed_dirs += 1

for p in SCAFFOLD.rglob("*.pyc"):
    try:
        p.unlink()
        removed_pyc += 1
    except FileNotFoundError:
        pass

for name in [
    "config", "tools", "prompt", "backends",
    "agent", "guardrails", "harness", "run_eval"
]:
    sys.modules.pop(name, None)

print("PASS: local Python caches cleared.")
print("__pycache__ folders removed:", removed_dirs)
print(".pyc files removed          :", removed_pyc)



## Step 7 — Import the project fresh and verify the active configuration

**Stop here if anything is wrong.** Do not run the paid battery until the model, version, tool-call mode, backend, prices, problem, and data path are correct.


In [ ]:

import os
import sys
import importlib
from pathlib import Path

os.chdir(SCAFFOLD)
if str(SCAFFOLD) not in sys.path:
    sys.path.insert(0, str(SCAFFOLD))

for name in [
    "config", "tools", "prompt", "backends",
    "agent", "guardrails", "harness"
]:
    sys.modules.pop(name, None)

import config
import harness

print("=" * 72)
print("ACTIVE D5 CONFIGURATION")
print("=" * 72)
print(config.summary())
print("TOOL_CALL_MODE =", config.TOOL_CALL_MODE)
print("PRICE_IN       =", config.PRICE_IN, "USD / 1M input tokens")
print("PRICE_OUT      =", config.PRICE_OUT, "USD / 1M output tokens")
print("DATA_ROOT      =", config.data_root())
print("=" * 72)

assert config.BACKEND == "live"
assert config.PROBLEM == "B"
assert config.MODEL == MODEL
assert config.VERSION == RUN_VERSION
assert config.TOOL_CALL_MODE == TOOL_CALL_MODE
assert float(config.PRICE_IN) == float(PRICE_IN)
assert float(config.PRICE_OUT) == float(PRICE_OUT)
assert Path(config.data_root()).resolve() == REFERENCE.resolve()

print("PASS: active configuration matches the requested run.")



## Step 8 — Free verification of the frozen 40-case / 56-trial battery

This does not call the live model.


In [ ]:

all_cases = harness.load_cases("B")
key = harness.load_key("B")

eval_cases = [c for c in all_cases if c.startswith("REF-EV")]
expected_cases = [f"REF-EV{i:03d}" for i in range(1, 41)]

negative_cases = [
    c for c in eval_cases
    if harness._is_negative(key[c])
]
ordinary_cases = [
    c for c in eval_cases
    if not harness._is_negative(key[c])
]

trial_count = len(ordinary_cases) + 3 * len(negative_cases)

checks = {
    "55 total referrals": len(all_cases) == 55,
    "40 frozen evaluation cases": len(eval_cases) == 40,
    "IDs exactly REF-EV001..REF-EV040": eval_cases == expected_cases,
    "32 ordinary cases": len(ordinary_cases) == 32,
    "8 negative cases": len(negative_cases) == 8,
    "56 total trials": trial_count == 56,
}

for label, ok in checks.items():
    print(f"[{'PASS' if ok else 'FAIL'}] {label}")

if not all(checks.values()):
    raise RuntimeError("Frozen battery verification failed. Do not run live.")

print("\nREADY: frozen 40-case set -> 56 live trials")



## Step 9 — Print the exact prompt/config before spending money

`--prompt` is free. It lets you confirm the actual prompt version being sent to the model.

For normal D5 members, the printed run header should show `VERSION=v2`. The assigned v1 comparison member should see `VERSION=v1`.


In [ ]:

import subprocess
import sys

prompt_check = subprocess.run(
    [sys.executable, "run_eval.py", "--prompt"],
    cwd=str(SCAFFOLD),
    env=os.environ.copy(),
    text=True,
)

if prompt_check.returncode != 0:
    raise RuntimeError("Prompt/config check failed.")

print("\nPASS: prompt command completed without a live battery.")



## Step 10 — Run the paid 56-trial live battery and measure total runtime

This is the cell that costs money.

The notebook records **wall-clock time** from immediately before `run_eval.py --battery` starts until it finishes. Runtime varies by model/provider. A prior Gemini 2.5 Flash Lite run took roughly a few minutes; use the measured value from your own run in your records.


In [ ]:

import subprocess
import sys
import time
from datetime import datetime, timezone

print("Starting paid live battery...")
print("Model :", MODEL)
print("Version:", RUN_VERSION)
print("Mode  :", TOOL_CALL_MODE)
print()

started_utc = datetime.now(timezone.utc).isoformat()
t0 = time.perf_counter()

completed = subprocess.run(
    [sys.executable, "run_eval.py", "--battery"],
    cwd=str(SCAFFOLD),
    env=os.environ.copy(),
    text=True,
)

elapsed_seconds = time.perf_counter() - t0
elapsed_minutes = elapsed_seconds / 60.0
finished_utc = datetime.now(timezone.utc).isoformat()

print()
print("=" * 72)
print("BATTERY WALL-CLOCK RUNTIME")
print("=" * 72)
print(f"Elapsed seconds : {elapsed_seconds:.1f}")
print(f"Elapsed minutes : {elapsed_minutes:.2f}")
print("=" * 72)

if completed.returncode != 0:
    raise RuntimeError(
        "The battery did not finish successfully. "
        "Do not treat an incomplete result as the final model measurement."
    )



## Step 11 — Read and verify the result file

The raw live result is written by the project as:

```text
A2_scaffold/results_d5_live.json
```

This cell checks that it contains exactly 56 trials and prints the summary.


In [ ]:

import json
from pathlib import Path

raw_result = SCAFFOLD / "results_d5_live.json"

if not raw_result.exists():
    raise FileNotFoundError(
        "results_d5_live.json was not created."
    )

data = json.loads(raw_result.read_text(encoding="utf-8"))
summary = data.get("summary", {})

if summary.get("trials") != 56:
    raise RuntimeError(
        f"Expected 56 completed trials, got {summary.get('trials')!r}."
    )

print("=" * 72)
print("D5 RESULT SUMMARY")
print("=" * 72)
for k, v in summary.items():
    print(f"{k:24s}: {v}")
print("=" * 72)
print("PASS: complete 56-trial result found.")



## Step 12 — Save a model-specific result copy with reproducibility metadata

The original `results_d5_live.json` is left unchanged.

This cell creates a second file in:

```text
<PACKAGE_ROOT>/D5_results/
```

The filename includes the member, model, version, and tool-call mode. The copy also records:
- member name
- model
- input/output prices used
- version
- serial/parallel mode
- measured wall-clock runtime
- UTC start/end time
- SHA-256 hashes of the main code files

This prevents teammates' result files from overwriting one another.


In [ ]:

import hashlib
import json
import re
from pathlib import Path

RESULTS_DIR = PACKAGE_ROOT / "D5_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def safe_name(s):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(s)).strip("_")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

archive = json.loads(raw_result.read_text(encoding="utf-8"))

archive["runner_metadata"] = {
    "member_name": MEMBER_NAME,
    "model": MODEL,
    "price_in_usd_per_1m": PRICE_IN,
    "price_out_usd_per_1m": PRICE_OUT,
    "version": RUN_VERSION,
    "tool_call_mode": TOOL_CALL_MODE,
    "backend": "live",
    "problem": "B",
    "elapsed_seconds": round(elapsed_seconds, 3),
    "elapsed_minutes": round(elapsed_minutes, 3),
    "started_utc": started_utc,
    "finished_utc": finished_utc,
    "raw_result_file": "A2_scaffold/results_d5_live.json",
    "code_sha256": {
        name: sha256_file(SCAFFOLD / name)
        for name in [
            "config.py",
            "prompt.py",
            "agent.py",
            "backends.py",
            "harness.py",
            "run_eval.py",
            "tools.py",
            "guardrails.py",
        ]
    },
}

archive_name = (
    f"d5_{safe_name(MEMBER_NAME)}_"
    f"{safe_name(MODEL)}_"
    f"{RUN_VERSION}_{TOOL_CALL_MODE}.json"
)

archive_path = RESULTS_DIR / archive_name
archive_path.write_text(
    json.dumps(archive, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Saved raw result     :", raw_result)
print("Saved archived result:", archive_path)
print()
print("SEND THE ARCHIVED RESULT FILE TO THE TEAM.")



## Optional final step — Restore `config.py` to scripted mode

The submitted repository should default to the free scripted backend.

If you are working in your own disposable D5 package copy, restoring is optional. If this copy may later be used for submission, restore the pre-run `config.py` after saving the live result.


In [ ]:

# OPTIONAL: uncomment these lines only AFTER the live result has been saved.

# import shutil
# backup_path = SCAFFOLD / "config.py.before_d5_runner"
# if backup_path.exists():
#     shutil.copy2(backup_path, SCAFFOLD / "config.py")
#     print("Restored config.py from pre-run backup.")
# else:
#     print("No backup found. Set BACKEND='scripted' manually before submission.")
